# 01 · The Python Data Model & How Attribute Lookup Really Works

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 60 min

> This module goes **under the hood**. It assumes Modules 05, 06 and 09 (functions, OOP, decorators). Nothing here is in the 📗 bootcamp or 📘 handbook.

### 🎯 What you will learn
- "Everything is an object": identity, type and value
- How `type` and `object` relate (the root of everything)
- Namespaces: instance `__dict__`, class `__dict__`, the MRO
- The **exact attribute lookup algorithm** (and why descriptors exist)
- `__getattribute__` vs `__getattr__`, `__setattr__`, `__delattr__`, `__dir__`
- How methods become **bound methods**
- Special methods are looked up on the **type**, not the instance
- The complete map of Python's **protocols** (`__format__`, `__index__`, `__fspath__`, `__class_getitem__`, ...)

## 1. Every object has an identity, a type and a value

In [1]:
x = [1, 2, 3]
print(id(x))          # identity: never changes during the object's life
print(type(x))        # type: decides what operations the object supports
print(x)              # value: may change (if the type is mutable)

for obj in [42, "hi", len, print, list, type, None, ..., NotImplemented]:
    print(f"{obj!r:<35} is an instance of {type(obj).__name__}")

4468584256
<class 'list'>
[1, 2, 3]
42                                  is an instance of int
'hi'                                is an instance of str
<built-in function len>             is an instance of builtin_function_or_method
<built-in function print>           is an instance of builtin_function_or_method
<class 'list'>                      is an instance of type
<class 'type'>                      is an instance of type
None                                is an instance of NoneType
Ellipsis                            is an instance of ellipsis
NotImplemented                      is an instance of NotImplementedType


Functions, classes, modules and types are **objects too**: you can pass them around, store them, and give them attributes.

In [2]:
def greet():
    return "hi"
greet.calls = 0                 # functions have a __dict__ too!
print(greet.__dict__, greet.__name__, greet.__code__.co_name)
import math
print(type(math), math.__dict__["pi"])

{'calls': 0} greet greet
<class 'module'> 3.141592653589793


## 2. `type` and `object`: the two roots

- Every class ultimately **inherits from** `object`
- Every class is an **instance of** `type` (its *metaclass*, lesson 03)
- `type` is an instance of itself, and `type` inherits from `object`

```
            instance of              instance of
   x ─────────────────► list ─────────────────► type ◄─┐
                         │                       │ └───┘ (type is an instance of type)
                    subclass of             subclass of
                         ▼                       ▼
                       object ◄───────────────── ┘
```

In [3]:
print(isinstance(list, type), issubclass(list, object))
print(type(type), type(object), issubclass(type, object), isinstance(object, type))
print(int.__mro__, bool.__mro__)

True True
<class 'type'> <class 'type'> True True
(<class 'int'>, <class 'object'>) (<class 'bool'>, <class 'int'>, <class 'object'>)


## 3. Namespaces: where attributes live

In [4]:
class Dog:
    species = "Canis familiaris"            # stored in the CLASS __dict__
    def __init__(self, name):
        self.name = name                    # stored in the INSTANCE __dict__
    def bark(self):
        return f"{self.name}: Woof"

rex = Dog("Rex")
print("instance:", rex.__dict__)
print("class   :", {k: v for k, v in Dog.__dict__.items() if not k.startswith("__")})
print(type(Dog.__dict__))                   # a read-only mappingproxy
rex.__dict__["age"] = 3                     # attributes ARE dictionary entries
print(rex.age)

instance: {'name': 'Rex'}
class   : {'species': 'Canis familiaris', 'bark': <function Dog.bark at 0x10a5ac1a0>}
<class 'mappingproxy'>
3


## 4. The attribute lookup algorithm (`obj.attr`)

`object.__getattribute__(obj, "attr")` does this:

1. Find `attr` in the **type's MRO** (`type(obj).__mro__`). If it's a **data descriptor** (defines `__set__` or `__delete__`, e.g. `property`), call its `__get__` → **done**
2. Else, if `attr` is in the **instance `__dict__`** → return it
3. Else, if found in the type in step 1: if it's a **non-data descriptor** (only `__get__`, e.g. a function), call `__get__` (this is how **bound methods** are made); otherwise return it as is (plain class attribute)
4. Else raise `AttributeError` → Python then calls **`__getattr__`** if the class defines it

```
obj.attr ──► data descriptor on type? ──yes──► descriptor.__get__(obj, type)
                    │no
                    ▼
            in obj.__dict__? ──yes──► return it
                    │no
                    ▼
            on type (MRO)? ──yes──► non-data descriptor? → __get__ ; else the value
                    │no
                    ▼
            __getattr__ defined? ──yes──► __getattr__(obj, "attr")
                    │no
                    ▼
              AttributeError
```

In [5]:
class Demo:
    cls_attr = "from class"

    @property
    def prop(self):                      # property = DATA descriptor
        return "from property"

d = Demo()
d.__dict__["cls_attr"] = "from instance"     # shadows the class attribute (step 2 beats step 3)
d.__dict__["prop"] = "from instance"         # does NOT shadow the property (step 1 beats step 2)
print(d.cls_attr, "|", d.prop)

from instance | from property


### Bound methods are made by descriptors

In [6]:
f = Dog.__dict__["bark"]                 # the raw function stored in the class
print(type(f))
bound = f.__get__(rex, Dog)              # what `rex.bark` does behind the scenes
print(bound, bound.__self__ is rex, bound.__func__ is f)
print(bound(), rex.bark())
print(Dog.bark(rex))                     # calling through the class: pass self explicitly

<class 'function'>
<bound method Dog.bark of <__main__.Dog object at 0x10a57d6a0>> True True
Rex: Woof Rex: Woof
Rex: Woof


## 5. Customising attribute access

| Hook | Called when |
|---|---|
| `__getattribute__(self, name)` | **every** attribute read (override with great care) |
| `__getattr__(self, name)` | only when normal lookup **fails** |
| `__setattr__(self, name, value)` | every assignment `obj.x = v` |
| `__delattr__(self, name)` | `del obj.x` |
| `__dir__(self)` | `dir(obj)` (helps autocompletion) |

In [7]:
class Tracked:
    def __init__(self):
        object.__setattr__(self, "_log", [])        # bypass our own __setattr__

    def __getattribute__(self, name):
        if not name.startswith("_"):
            object.__getattribute__(self, "_log").append(f"get {name}")
        return object.__getattribute__(self, name)  # ✅ delegate to object, never self.name (infinite recursion!)

    def __getattr__(self, name):                     # fallback for missing attributes
        return f"<no attribute {name!r}>"

    def __setattr__(self, name, value):
        self._log.append(f"set {name}={value!r}")
        object.__setattr__(self, name, value)

t = Tracked()
t.color = "red"
print(t.color, "|", t.size)
print(t._log)

red | <no attribute 'size'>
["set color='red'", 'get color', 'get size']


### A practical proxy: lazy loading + delegation

In [8]:
class LazyProxy:
    """Create the real object only when it's first used."""
    def __init__(self, factory):
        object.__setattr__(self, "_factory", factory)
        object.__setattr__(self, "_obj", None)

    def _target(self):
        if self._obj is None:
            print("  (creating the expensive object now)")
            object.__setattr__(self, "_obj", self._factory())
        return self._obj

    def __getattr__(self, name):              # anything not found on the proxy → the real object
        return getattr(self._target(), name)

config = LazyProxy(lambda: {"debug": True, "db": "postgres"})
print("proxy created")
print(config.get("db"), config.keys())

proxy created
  (creating the expensive object now)
postgres dict_keys(['debug', 'db'])


## 6. Special methods are looked up on the **type**

For speed and consistency, implicit calls like `len(obj)`, `obj + 1`, `str(obj)` look up `__len__`, `__add__`, `__str__` on **`type(obj)`**, skipping the instance and `__getattribute__`:

In [9]:
class Box:
    def __len__(self):
        return 1

b = Box()
b.__len__ = lambda: 99              # set on the INSTANCE
print(len(b), b.__len__())          # len() ignores it → 1 ; explicit call uses it → 99

1 99


## 7. The protocol map

The "data model" is the set of special methods that let your objects plug into Python syntax:

| Protocol | Methods | Enables |
|---|---|---|
| Representation | `__repr__`, `__str__`, `__format__`, `__bytes__` | `repr()`, `print()`, `f"{x:spec}"`, `bytes()` |
| Truth / numbers | `__bool__`, `__int__`, `__float__`, `__index__`, `__round__`, `__abs__` | `if x:`, conversions, `lst[x]`, `round()` |
| Arithmetic | `__add__`/`__radd__`/`__iadd__`, `__matmul__`, `__neg__`... | operators |
| Comparison / hashing | `__eq__`, `__lt__`..., `__hash__` | `==`, `<`, sets, dict keys |
| Container | `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__iter__`, `__reversed__`, `__missing__` | `len`, `[]`, `in`, `for` |
| Iterator | `__iter__`, `__next__` | iteration |
| Callable | `__call__` | `obj()` |
| Context manager | `__enter__`, `__exit__` (`__aenter__`, `__aexit__`) | `with` / `async with` |
| Async | `__await__`, `__aiter__`, `__anext__` | `await`, `async for` |
| Attributes | `__getattr__`, `__getattribute__`, `__setattr__`, `__delattr__`, `__dir__` | `obj.x` |
| Descriptors | `__get__`, `__set__`, `__delete__`, `__set_name__` | properties, methods (lesson 02) |
| Class creation | `__new__`, `__init__`, `__init_subclass__`, `__class_getitem__`, `__prepare__` | lesson 03 |
| Copy / pickle | `__copy__`, `__deepcopy__`, `__getstate__`, `__setstate__`, `__reduce__` | lesson 05 |
| Filesystem / buffers | `__fspath__`, `__buffer__` (3.12) | `open(obj)`, `memoryview(obj)` |

### A few lesser-known protocols in action

In [10]:
import os
from datetime import date

class Money:
    def __init__(self, amount, currency="INR"):
        self.amount, self.currency = amount, currency
    def __format__(self, spec):                      # f"{m:...}"
        if spec == "short":
            return f"{self.amount:,.0f} {self.currency}"
        return f"{self.currency} {format(self.amount, spec or ',.2f')}"
    def __bool__(self):
        return self.amount != 0

class Slot:
    def __init__(self, n): self.n = n
    def __index__(self): return self.n               # usable as an index / in range() / bin()

class ProjectPath:
    def __init__(self, *parts): self.parts = parts
    def __fspath__(self): return os.path.join(*self.parts)   # accepted by open(), os.path...

m = Money(1234567.891)
print(f"{m}", "|", f"{m:short}", "|", f"{m:.1f}", "|", bool(Money(0)))
print(["a", "b", "c"][Slot(1)], list(range(Slot(3))), bin(Slot(5)))
print(os.fspath(ProjectPath("data", "2026", "report.csv")), os.path.exists(ProjectPath("nope")))

INR 1,234,567.89 | 1,234,568 INR | INR 1234567.9 | False
b [0, 1, 2] 0b101
data/2026/report.csv False


`__missing__` (dict subclasses) and `__class_getitem__` (generic syntax `MyBox[int]`):

In [11]:
class DefaultZero(dict):
    def __missing__(self, key):            # called by dict.__getitem__ for missing keys
        return 0

counts = DefaultZero()
counts["a"] += 1
print(counts, counts["zzz"])

class Box:
    def __class_getitem__(cls, item):      # makes Box[int] work (used by typing generics)
        return f"{cls.__name__} of {item.__name__}"

print(Box[int])

{'a': 1} 0
Box of int


## 📝 Summary
- Every value is an object with identity, type and value; classes are instances of `type`.
- Attributes live in instance and class `__dict__`s; lookup order: **data descriptors → instance dict → non-data descriptors/class attrs → `__getattr__`**.
- Functions are descriptors: that's how `self` gets bound.
- Implicit special-method calls look on the type, not the instance.
- The data model's protocols let your classes behave like built-ins.

## 🏋️ Practice
1. Write a `ReadOnlyView` proxy that forwards attribute *reads* to a wrapped object but raises `AttributeError` on any assignment.
2. Write `AttrDict(dict)` that allows `d.key` access for existing keys (`__getattr__`) and `d.key = v` assignment.
3. Implement `__format__` for a `Temperature` class supporting `f"{t:C}"` and `f"{t:F}"`.

<details><summary>Show solutions</summary>

```python
class ReadOnlyView:
    def __init__(self, target):
        object.__setattr__(self, "_target", target)
    def __getattr__(self, name):
        return getattr(self._target, name)
    def __setattr__(self, name, value):
        raise AttributeError("read-only view")

class AttrDict(dict):
    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name) from None
    def __setattr__(self, name, value):
        self[name] = value

class Temperature:
    def __init__(self, c): self.c = c
    def __format__(self, spec):
        return f"{self.c * 9 / 5 + 32:.1f}°F" if spec == "F" else f"{self.c:.1f}°C"

print(f"{Temperature(25):C} {Temperature(25):F}")
d = AttrDict(a=1); d.b = 2; print(d.a, d.b, d)
```
</details>